# 06 — Interpretability: Time-Point and Electrode Feature Importance

This notebook adds an interpretability analysis to the selected OpenNeuro ERP/oddball stimulus-classification pipeline.

## Purpose

The main goal is to connect the current classification stage back to the original thesis motivation of identifying meaningful EEG structure. To do this, we extract and visualize feature importance from the strongest classical models.

## What this notebook will produce

For **LDA** and **Logistic Regression**, this notebook produces dataset-level interpretability outputs:

- a **time-point importance curve**
- an **electrode importance summary**
- an **electrode topoplot** in the P300 window
- a **two-panel report-ready figure**

## Main idea

For each subject, we:

1. load the same cropped benchmark dataset used in the main within-subject benchmark
2. fit a classical model on the subject's full cropped data
3. extract the model coefficient vector
4. reshape it into `(channels, time)`
5. summarize:
   - **time importance** = mean absolute coefficient across electrodes
   - **electrode importance** = mean absolute coefficient across time
   - **P300 topographic importance** = mean absolute coefficient within the P300 window

These plots are interpretability diagnostics. They are **not** new benchmark metrics.

All outputs are saved under:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/tables/interpretability/broadband/`

and

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/figures/interpretability/broadband/`

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.features import flatten_epochs
from oddball_eeg.io import save_dataframe, save_experiment_metadata
from oddball_eeg.loader import load_subjects_grouped, make_subject_summary_table
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)

## 1. Load the main benchmark configuration

We use the same configuration as the main within-subject benchmark so that the interpretability analysis remains directly aligned with the benchmark results already used in the report.

By default, this notebook uses the same subjects and the same cropped benchmark window as the main experiment.

If needed, the subject list can be narrowed later for debugging or for a more focused figure.

In [ ]:
set_pandas_display_options()

# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds003061_within_subject_main"
# EXPERIMENT_NAME = "ds005863_within_subject_main"
EXPERIMENT_NAME = "ds006480_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)
cfg = experiment_family["main"]

# By default, use the same subjects as the main benchmark.
SUBJECTS_TO_USE = list(cfg.subjects.subjects)

# Models to explain
MODELS_TO_EXPLAIN = ("lda", "logreg")

# Choose which model's two-panel figure will be used in the report
PRIMARY_MODEL_FOR_REPORT = "lda"

# ERP/P300 window used for topographic importance.
# For non-P300 ERP datasets later, rename these constants if needed,
# but keep the same idea: summarize importance in the expected ERP window.
P300_TMIN = 0.25
P300_TMAX = 0.50

OUT_TABLE_DIR = cfg.table_dir("interpretability", "broadband")
OUT_FIG_DIR = cfg.figure_dir("interpretability", "broadband")

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "06_interpretability_feature_importance.ipynb",
        "purpose": "Classical-model coefficient interpretability: time and electrode importance.",
        "experiment_key": EXPERIMENT_NAME,
        "models_to_explain": list(MODELS_TO_EXPLAIN),
        "primary_model_for_report": PRIMARY_MODEL_FOR_REPORT,
        "p300_window": [P300_TMIN, P300_TMAX],
        "table_dir": str(OUT_TABLE_DIR),
        "figure_dir": str(OUT_FIG_DIR),
    },
)

print("Experiment key:", EXPERIMENT_NAME)
print("Experiment    :", cfg.name)
print("Dataset       :", cfg.dataset.dataset)
print("Task          :", cfg.dataset.task)
print("Signal config :", cfg.signal)
print("Subjects      :", SUBJECTS_TO_USE)
print("Crop          :", cfg.effective_crop_window())
print("Models        :", MODELS_TO_EXPLAIN)
print("Primary       :", PRIMARY_MODEL_FOR_REPORT)
print("Table dir     :", OUT_TABLE_DIR)
print("Figure dir    :", OUT_FIG_DIR)

## 2. Load the grouped dataset and apply the main benchmark crop window

This notebook reuses the same grouped chronological loading logic and the same crop window used in the main within-subject benchmark.

That keeps the interpretability plots aligned with the actual benchmark setting already reported elsewhere in the thesis.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=SUBJECTS_TO_USE,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

crop_window = cfg.effective_crop_window()
assert crop_window is not None, "Expected an active crop window for the main benchmark."

data_main = apply_crop_to_loaded_dataset(
    data_full,
    tmin=crop_window[0],
    tmax=crop_window[1],
)

subject_summary_main = make_subject_summary_table(data_main)
subject_summary_main = round_numeric_columns(subject_summary_main, decimals=4)

display(subject_summary_main)

save_dataframe(
    subject_summary_main,
    OUT_TABLE_DIR / "loaded_subject_summary_main_cropped.csv",
)

## 3. Define helper functions for coefficient extraction and importance summaries

The classical models are implemented as sklearn pipelines. After fitting them, we extract the coefficient vector from the final classifier, reshape it into `(channels, time)`, and then summarize its magnitude.

We use **absolute coefficient magnitude** because the sign itself is less important here than the overall strength and location of discriminative information.

We also normalize each subject-level coefficient map before averaging across subjects so that one subject does not dominate the final aggregate figure.

In [ ]:
def extract_pipeline_coef(clf) -> np.ndarray:
    """
    Extract the coefficient vector from a fitted sklearn Pipeline.

    Expected classical models:
    - LDA pipeline
    - Logistic Regression pipeline
    """
    final_est = clf.named_steps["clf"]

    if not hasattr(final_est, "coef_"):
        raise ValueError(
            f"Final estimator {type(final_est).__name__} has no coef_ attribute."
        )

    coef = np.asarray(final_est.coef_, dtype=np.float64)

    # Binary classification should produce shape (1, n_features)
    if coef.ndim == 2:
        if coef.shape[0] != 1:
            raise ValueError(f"Expected binary coef shape (1, n_features), got {coef.shape}")
        coef = coef[0]

    return coef.astype(np.float32)


def fit_subject_model_and_get_coef(
        X: np.ndarray,
        y: np.ndarray,
        model_name: str,
        n_ch: int,
        n_times: int,
) -> np.ndarray:
    """
    Fit one classical model on the full subject dataset and return a
    coefficient map of shape (n_ch, n_times).
    """
    F = flatten_epochs(X)
    clf = get_model(model_name)
    clf.fit(F, y)

    coef_flat = extract_pipeline_coef(clf)
    coef_map = coef_flat.reshape(n_ch, n_times)

    return coef_map.astype(np.float32)


def normalize_importance_map(coef_map: np.ndarray) -> np.ndarray:
    """
    Convert a coefficient map into a normalized nonnegative importance map.

    We use absolute coefficients and normalize each subject/model map
    by its maximum value to reduce domination by one subject.
    """
    imp = np.abs(coef_map).astype(np.float32)
    denom = float(imp.max()) + 1e-12
    imp = imp / denom
    return imp

## 4. Define helper functions for topographic plotting

To make an electrode topoplot, we create an MNE `Info` object with a standard 10–20 montage and then keep only channels that have valid spatial coordinates.

This makes the notebook more robust in case some channels do not have usable positions for scalp plotting.

In [ ]:
def build_mne_info(ch_names: list[str], sfreq: float) -> mne.Info:
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    info.set_montage("standard_1020", on_missing="ignore")
    return info

## 5. Prepare the shared dimensions and the P300 mask

We extract the common time axis, channel names, and sampling frequency from the cropped benchmark dataset, and we define the P300 window that will be used for the topographic importance summary.

In [ ]:
times = data_main.times
ch_names = data_main.ch_names
sfreq = data_main.sfreq

n_ch = len(ch_names)
n_times = len(times)

p300_mask = (times >= P300_TMIN) & (times <= P300_TMAX)
if p300_mask.sum() == 0:
    raise ValueError("P300 window is empty after cropping.")

print("n_channels :", n_ch)
print("n_times    :", n_times)
print("time range  :", float(times[0]), "to", float(times[-1]))
print("P300 samples:", int(p300_mask.sum()))

## 6. Fit the classical models subject by subject and build subject-level importance tables

For each subject and each classical model:

- fit the model on the full cropped subject data
- extract the coefficient map
- normalize the importance map
- compute:
  - time importance
  - full-window electrode importance
  - P300-window electrode importance

We then save these subject-level values so they can be inspected later if needed.

In [ ]:
all_rows_time = []
all_rows_elec = []
subject_maps = {m: [] for m in MODELS_TO_EXPLAIN}

for sub in data_main.subjects:
    X = data_main.X_by_sub[sub]
    y = data_main.y_by_sub[sub]

    print("=" * 70)
    print(f"Subject {sub}")

    for model_name in MODELS_TO_EXPLAIN:
        coef_map = fit_subject_model_and_get_coef(
            X=X,
            y=y,
            model_name=model_name,
            n_ch=n_ch,
            n_times=n_times,
        )

        imp_map = normalize_importance_map(coef_map)
        subject_maps[model_name].append(imp_map)

        # Mean absolute importance across electrodes -> time profile
        time_imp = imp_map.mean(axis=0)

        # Mean absolute importance across time -> electrode profile
        elec_imp_full = imp_map.mean(axis=1)

        # P300-window electrode importance
        elec_imp_p300 = imp_map[:, p300_mask].mean(axis=1)

        for t_idx, t in enumerate(times):
            all_rows_time.append({
                "subject": sub,
                "model": model_name,
                "time_s": float(t),
                "importance": float(time_imp[t_idx]),
            })

        for ch_idx, ch in enumerate(ch_names):
            all_rows_elec.append({
                "subject": sub,
                "model": model_name,
                "channel": ch,
                "importance_full": float(elec_imp_full[ch_idx]),
                "importance_p300": float(elec_imp_p300[ch_idx]),
            })

df_time_subject = pd.DataFrame(all_rows_time)
df_elec_subject = pd.DataFrame(all_rows_elec)

save_dataframe(
    df_time_subject,
    OUT_TABLE_DIR / "subject_time_importance.csv",
)

save_dataframe(
    df_elec_subject,
    OUT_TABLE_DIR / "subject_electrode_importance.csv",
)

print("Saved:")
print("-", OUT_TABLE_DIR / "subject_time_importance.csv")
print("-", OUT_TABLE_DIR / "subject_electrode_importance.csv")

## 7. Aggregate the importance summaries across subjects

Now we average the subject-level importance values to create report-ready summaries.

These aggregate tables are what we will use for the final time-importance curve and topographic plot.

In [ ]:
df_time_mean = (
    df_time_subject
    .groupby(["model", "time_s"], as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
    )
)

df_elec_mean = (
    df_elec_subject
    .groupby(["model", "channel"], as_index=False)
    .agg(
        importance_full_mean=("importance_full", "mean"),
        importance_full_std=("importance_full", "std"),
        importance_p300_mean=("importance_p300", "mean"),
        importance_p300_std=("importance_p300", "std"),
    )
)

save_dataframe(
    df_time_mean,
    OUT_TABLE_DIR / "mean_time_importance.csv",
)

save_dataframe(
    df_elec_mean,
    OUT_TABLE_DIR / "mean_electrode_importance.csv",
)

print("Saved:")
print("-", OUT_TABLE_DIR / "mean_time_importance.csv")
print("-", OUT_TABLE_DIR / "mean_electrode_importance.csv")

display(
    round_numeric_columns(df_time_mean.head(), decimals=4)
)

## 8. Define the plotting functions

We create three kinds of visualizations:

1. **time-point importance curve**
2. **electrode topomap**
3. **two-panel report figure** combining the time curve and the P300 topomap

The two-panel figure is the one most suitable for direct insertion into the progress report.

In [ ]:
def plot_time_importance(
        df_time_mean: pd.DataFrame,
        model_name: str,
        p300_tmin: float,
        p300_tmax: float,
):
    sub_df = (
        df_time_mean[df_time_mean["model"] == model_name]
        .sort_values("time_s")
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sub_df["time_s"], sub_df["importance_mean"], linewidth=2)
    ax.axvline(0.0, linestyle="--", alpha=0.7)
    ax.axvspan(p300_tmin, p300_tmax, alpha=0.15)

    ax.set_title(
        f"{model_name.upper()} time-point importance — {cfg.dataset.dataset}, {cfg.dataset.task}"
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean normalized |coefficient|")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    return fig


def plot_electrode_topomap(
        df_elec_mean: pd.DataFrame,
        model_name: str,
        ch_names: list[str],
        sfreq: float,
        use_p300: bool = True,
):
    value_col = "importance_p300_mean" if use_p300 else "importance_full_mean"

    sub_df = (
        df_elec_mean[df_elec_mean["model"] == model_name]
        .set_index("channel")
        .reindex(ch_names)
        .reset_index()
    )

    values = sub_df[value_col].to_numpy(dtype=float)

    info = build_mne_info(ch_names=ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)
    info_valid = mne.pick_info(info, picks, copy=True)
    values_valid = values[picks]

    fig, ax = plt.subplots(figsize=(5, 4))
    mne.viz.plot_topomap(
        values_valid,
        info_valid,
        axes=ax,
        show=False,
        contours=6,
        cmap="viridis",
    )

    title_suffix = "P300 window" if use_p300 else "full window"
    ax.set_title(
        f"{model_name.upper()} electrode importance ({title_suffix})\n"
        f"{cfg.dataset.dataset}, {cfg.dataset.task}"
    )
    fig.tight_layout()
    return fig


def plot_two_panel_report_figure(
        df_time_mean: pd.DataFrame,
        df_elec_mean: pd.DataFrame,
        model_name: str,
        ch_names: list[str],
        sfreq: float,
        p300_tmin: float,
        p300_tmax: float,
):
    sub_df_t = (
        df_time_mean[df_time_mean["model"] == model_name]
        .sort_values("time_s")
        .reset_index(drop=True)
    )

    sub_df_e = (
        df_elec_mean[df_elec_mean["model"] == model_name]
        .set_index("channel")
        .reindex(ch_names)
        .reset_index()
    )

    values = sub_df_e["importance_p300_mean"].to_numpy(dtype=float)

    info = build_mne_info(ch_names=ch_names, sfreq=sfreq)
    picks = valid_topomap_picks(info)
    info_valid = mne.pick_info(info, picks, copy=True)
    values_valid = values[picks]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left panel: time importance
    axes[0].plot(sub_df_t["time_s"], sub_df_t["importance_mean"], linewidth=2)
    axes[0].axvline(0.0, linestyle="--", alpha=0.7)
    axes[0].axvspan(p300_tmin, p300_tmax, alpha=0.15)
    axes[0].set_title(f"{model_name.upper()} time-point importance")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Mean normalized |coefficient|")
    axes[0].grid(True, alpha=0.3)

    # Right panel: topomap
    mne.viz.plot_topomap(
        values_valid,
        info_valid,
        axes=axes[1],
        show=False,
        contours=6,
        cmap="viridis",
    )
    axes[1].set_title(
        f"{model_name.upper()} electrode importance\n"
        f"({P300_TMIN:.2f}–{P300_TMAX:.2f} s window)"
    )

    fig.tight_layout()
    return fig

## 9. Generate and save the figures

We now generate all plots for both classical models and save them to the interpretability output directory.

The most important output for the report is the **two-panel figure** for the selected primary model.

In [ ]:
for model_name in MODELS_TO_EXPLAIN:
    fig_time = plot_time_importance(
        df_time_mean=df_time_mean,
        model_name=model_name,
        p300_tmin=P300_TMIN,
        p300_tmax=P300_TMAX,
    )
    save_figure(
        fig_time,
        OUT_FIG_DIR / f"{model_name}_time_importance.png",
    )
    plt.show()

    fig_topo = plot_electrode_topomap(
        df_elec_mean=df_elec_mean,
        model_name=model_name,
        ch_names=ch_names,
        sfreq=sfreq,
        use_p300=True,
    )
    save_figure(
        fig_topo,
        OUT_FIG_DIR / f"{model_name}_electrode_topomap_p300.png",
    )
    plt.show()

    fig_two = plot_two_panel_report_figure(
        df_time_mean=df_time_mean,
        df_elec_mean=df_elec_mean,
        model_name=model_name,
        ch_names=ch_names,
        sfreq=sfreq,
        p300_tmin=P300_TMIN,
        p300_tmax=P300_TMAX,
    )
    save_figure(
        fig_two,
        OUT_FIG_DIR / f"{model_name}_report_two_panel.png",
    )
    plt.show()

print("Saved interpretability figures to:", OUT_FIG_DIR)

## 10. Inspect the most important electrodes for the primary report model

This table is useful if you want to mention the strongest contributing electrodes explicitly in your text or use it for a caption/check.

In [ ]:
top_electrodes = (
    df_elec_mean[df_elec_mean["model"] == PRIMARY_MODEL_FOR_REPORT]
    .sort_values("importance_p300_mean", ascending=False)
    .reset_index(drop=True)
)

display(
    round_numeric_columns(top_electrodes.head(15), decimals=4)
)

## 11. Save the primary electrode ranking and print the key output paths

This final cell saves the ranked electrode table and prints the most important figure paths so they are easy to locate for report insertion.

In [ ]:
save_dataframe(
    top_electrodes,
    OUT_TABLE_DIR / f"{PRIMARY_MODEL_FOR_REPORT}_top_electrodes_p300.csv",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "06_interpretability_feature_importance.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "models_to_explain": list(MODELS_TO_EXPLAIN),
        "primary_model_for_report": PRIMARY_MODEL_FOR_REPORT,
        "p300_window": [P300_TMIN, P300_TMAX],
        "table_dir": str(OUT_TABLE_DIR),
        "figure_dir": str(OUT_FIG_DIR),
        "n_subjects": int(len(data_main.subjects)),
        "n_channels": int(len(data_main.ch_names)),
        "n_times": int(len(data_main.times)),
    },
)

print("Primary report figure:")
print(OUT_FIG_DIR / f"{PRIMARY_MODEL_FOR_REPORT}_report_two_panel.png")

print("\nOther useful outputs:")
print(OUT_FIG_DIR / f"{PRIMARY_MODEL_FOR_REPORT}_time_importance.png")
print(OUT_FIG_DIR / f"{PRIMARY_MODEL_FOR_REPORT}_electrode_topomap_p300.png")
print(OUT_TABLE_DIR / "mean_time_importance.csv")
print(OUT_TABLE_DIR / "mean_electrode_importance.csv")
print(OUT_TABLE_DIR / f"{PRIMARY_MODEL_FOR_REPORT}_top_electrodes_p300.csv")